# Directly-follows graph: demo on the running example

Page: `docs/pages/directly-follows-graph.md` (status: suggestion, 2026-11-14); user-facing description: module note `overview/module/Directly-Follows Graph Page.md` (outside this repo).
Code: package `ocelescope_module_location_based_ocpm.dfg`; this notebook only imports and calls its functions, so a change in the package shows up here on the next run.
Further detail: the docstrings (e.g. `help(path_filter)`).

What the notebook does, one step per section:
1. loads the running example and derives the locations (for $HU$, $M$, $P$),
2. builds the object traces of the selected object types (`object_traces`),
3. applies the path filter (`path_filter`),
4. counts the directly-follows arcs, applies the arc filter and removes deselected activities (`directly_follows_arcs`, `arc_filter`, `remove_activities`),
5. computes the activity nodes (`shown_activities`, `activity_frequencies`, `involved_object_types`),
6. runs everything in one call, as the page does (`resolve_selection`, `directly_follows_graph`).

In [ ]:
import json
from pathlib import Path

import polars as pl
from ocelescope import OCEL

from ocelescope_module_location_based_ocpm.derivation.config import DerivationConfig
from ocelescope_module_location_based_ocpm.derivation.pipeline import derive
from ocelescope_module_location_based_ocpm.dfg.arcs import arc_filter, directly_follows_arcs, remove_activities
from ocelescope_module_location_based_ocpm.dfg.graph import (
    DfgRequest,
    TypeThresholds,
    default_selection,
    directly_follows_graph,
    log_frames,
    resolve_selection,
)
from ocelescope_module_location_based_ocpm.dfg.nodes import activity_frequencies, involved_object_types, shown_activities
from ocelescope_module_location_based_ocpm.dfg.traces import considered_objects, object_traces
from ocelescope_module_location_based_ocpm.dfg.variants import object_variants, path_filter, variant_ranking

EXAMPLE = Path.cwd().parent / "data" / "running_example"
pl.Config.set_tbl_rows(40)

## 1. Log, configuration and frames: `log_frames(ocel, derivation)`

The graph needs the e2o relations with object types, the events with the flags $M$, $T$, $P$ (for the AS4 tie order), all objects, the handling units $HU$ and the processing operations $P$. `log_frames` reads them from the log and its derivation into a `LogFrames` object.

In [ ]:
ocel = OCEL.read(EXAMPLE / "running_example.sqlite")
config = DerivationConfig.model_validate(json.loads((EXAMPLE / "configuration.json").read_text()))
tables = {"locations.csv": pl.read_csv(EXAMPLE / "inputs" / "locations.csv", infer_schema=False)}
derivation = derive(ocel, config, tables)
frames = log_frames(ocel, derivation)
frames.e2o

## 2. Traces: `considered_objects(objects, object_types, handling_units)`, `object_traces(e2o, events, objects, processing)`

| parameter | meaning |
|---|---|
| `object_types` | the selected object types (default on the page: `default_selection`, the types with handling units) |
| `handling_units` | $HU$; a type with handling units uses only its handling units, other types all their objects |
| `processing` | $P$ as event ids, for the AS4 tie rank (None = $T$) |

How it works: joins e2o (an object linked twice to an event counts once) with the objects and events and sorts by object, time, AS4 tie rank, event id (`visits.order.sort_by_seq`); `position` is the 1-based position in the object's trace.

In [ ]:
selection = default_selection(frames.objects, frames.handling_units.to_list())
types = [s.object_type for s in selection]
objects = considered_objects(frames.objects, types, frames.handling_units.to_list())
traces = object_traces(frames.e2o, frames.events, objects, processing=frames.processing.to_list())
traces

## 3. Path filter: `path_filter(traces, thresholds)`

The path (variant) of an object is its sequence of activities (`object_variants`). `variant_ranking` counts the objects per path and object type, most frequent first (ties alphabetically), with the objects covered by the paths ranked before (`covered_before`).
A path of type $ot$ is kept if `covered_before` $< \pi_{ot}/100 \cdot |O_{ot}|$; `thresholds` maps object type → $\pi_{ot}$ in percent (missing = 100, all paths; default on the page 30 %).
In the running example every coil has its own path, so 50 % keeps the first three paths (3 of 5 coils).

In [ ]:
variant_ranking(object_variants(traces))

In [ ]:
kept_traces = path_filter(traces, {"coil": 50})
kept_traces["object"].unique().sort().to_list()

## 4. Arcs, arc filter and deselected activities: `directly_follows_arcs(traces)`, `arc_filter(arcs, thresholds)`, `remove_activities(arcs, activities)`

Arcs per object type with `count` (traversals) and `objects` (distinct objects); a null `source` is the start node $\triangleright_{ot}$, a null `target` the end node $\square_{ot}$.
`arc_filter` keeps, per object type, the $\lceil \alpha_{ot}/100 \cdot n \rceil$ most frequent of its $n$ arcs between two activities (by `count`, then `objects`, then source and target) and always all start and end arcs; `thresholds` maps object type → $\alpha_{ot}$ in percent (missing = 100, all arcs; default on the page 10 %).
`remove_activities` drops every arc that touches a deselected activity (no projection: no new arc between its neighbours).

In [ ]:
arcs = directly_follows_arcs(traces)
arcs

In [ ]:
kept_arcs = arc_filter(arcs, {"coil": 50})
kept_arcs

In [ ]:
remove_activities(kept_arcs, ["load truck"])

## 5. Activity nodes: `shown_activities(arcs)`, `activity_frequencies(traces, activities)`, `involved_object_types(traces, activities, typed_e2o)`

Shown: the activities touched by a kept arc. Frequency: the events of the activity in the (path-filtered) traces. Involved object types: per activity and object type (selected or not) the events and objects; the page draws the unselected ones as circles. The running example has only the handling-unit types coil and slab, so no circles appear here (`tests/test_dfg.py` has a log with a truck).

In [ ]:
activities = shown_activities(arcs)
activity_frequencies(traces, activities)

In [ ]:
involved_object_types(traces, activities, frames.e2o)

## 6. Everything at once: `resolve_selection(request, objects, handling_units)`, `directly_follows_graph(frames, selection=None, traces=None, excluded_activities=())`

What the page's route does: `resolve_selection` turns the page's request (`DfgRequest`: selected types or None for `default_selection`, general `path_threshold` and `arc_threshold`, own values per type in `type_thresholds`, `excluded_activities`) into one `TypeSetting(object_type, path_threshold, arc_threshold)` per selected type. `directly_follows_graph` returns all object types with what is kept of them, the activity nodes, the kept arcs and the activity list (`activity_options`, by events in the log, with `on_included_trace`). `traces` can be passed from a cache.

In [ ]:
request = DfgRequest(
    type_thresholds=[TypeThresholds(object_type="coil", path_threshold=100, arc_threshold=50)],
    excluded_activities=["load truck"],
)
selection = resolve_selection(request, frames.objects, frames.handling_units.to_list())
graph = directly_follows_graph(frames, selection, excluded_activities=request.excluded_activities)
pl.DataFrame([t.model_dump() for t in graph.object_types])

In [ ]:
pl.DataFrame([a.model_dump() for a in graph.arcs])